In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from natsort import natsorted
import yaml
import os
import subprocess

%matplotlib inline

In [ ]:
from libs.mapper.create_topomap import CostmapData
from libs.control.learnt_controller import ObjRelLearntController, visualize_prediction
from notebooks.viz_utils import plot_query_img_costmap_waypoints, plot_query_img_with_costmap

In [ ]:
SCENE_DIR = Path("./data/gs2_modified/img_L_1preF")
# SCENE_DIR = Path("./data/adamya/scene_000")

SCENE_IMG_DIR = SCENE_DIR / "images"
MAP_BASE_DIR = Path("./data/gs2_modified/img_L_1preF")
# MAP_BASE_DIR = Path(os.path.abspath("./outputs/carla_mapping"))

SCENE_NAME = "scene_000"
SCENE_MAP_DIR = MAP_BASE_DIR / SCENE_NAME

costmaps_file = SCENE_MAP_DIR / "costmaps_320x240_EC_NONE_NC_NONE_NCF_10.npz"

costmaps_data = np.load(costmaps_file)
# print("Costmaps data keys:", costmaps_data.files)
img_costmaps = costmaps_data["costmaps"]

img_paths = natsorted(SCENE_IMG_DIR.iterdir())

waypoints_save_path = MAP_BASE_DIR / "waypoints.npz"
waypoints = None


In [ ]:
output_imgs_dir = MAP_BASE_DIR / "outputs_viz"
if not output_imgs_dir.exists(): 
    os.makedirs(output_imgs_dir)

costmaps_viz_dir = MAP_BASE_DIR / "costmaps_viz"
if not costmaps_viz_dir.exists():
    os.makedirs(costmaps_viz_dir)

counter = 0
num_imgs = 100
for idx, path in enumerate(img_paths):
    # print(idx, path)
    img = cv2.cvtColor(cv2.imread(str(img_paths[idx])), cv2.COLOR_BGR2RGB)
    costmap  = img_costmaps[idx]
    plot_query_img_with_costmap(
        query_img=img,
        costmap=costmap,
        step_id=idx,
        save_path=costmaps_viz_dir / path.name
    )

    if num_imgs == counter:
        break

    counter += 1

In [ ]:
from hydra import compose, initialize
from omegaconf import OmegaConf

with initialize(version_base=None, config_path="configs/controller"):
    config = dict(compose(config_name="gs2_learnt"))
    # config = dict(compose(config_name="carla_learnt"))

print(OmegaConf.to_yaml(config))

In [ ]:
config['boost_final_goal'] = config.get('boost_final_goal', False)
controller = ObjRelLearntController(
    config=config,
    boost_final_goal=config['boost_final_goal']
)
controller

In [ ]:
controller.reset_params()

In [ ]:
waypoints = []

for idx, img_path in enumerate(img_paths):
    bgr = cv2.imread(str(img_path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)  # (H, W, 3) uint8
    # Corresponding costmap for this frame (pixel-wise path lengths)
    costmap = img_costmaps[idx]  # (H, W) float32
    # Run controller — goal_data is just the costmap (H, W)
    v, w, _ = controller.predict(rgb, costmap)
    waypoints.append(controller.action_pred.copy())
    # plt.imshow(rgb)
    # plt.show()
    # print(v, w, _)
    # break

waypoints = np.array(waypoints)  # (N_frames, len_traj_pred, 4)
print(f"Collected waypoints: {waypoints.shape}")  # e.g. (N, 10, 4)


In [ ]:
np.savez_compressed(
    waypoints_save_path,
    waypoints=waypoints,                           # (N, T, 4)
    img_paths=np.array([str(p) for p in img_paths]),  # (N,) string paths
)
print(f"Saved to {waypoints_save_path.resolve()}")


In [ ]:
output_imgs_dir = MAP_BASE_DIR / "outputs_viz"
if not output_imgs_dir.exists(): 
    os.makedirs(output_imgs_dir)

try:
    if waypoints is not None:    
        waypoints_data = np.load(waypoints_save_path)
        waypoints = waypoints_data['waypoints']
except FileNotFoundError:
    raise FileNotFoundError("Please generate the waypoints first, before running this code")

for idx, path in enumerate(img_paths):
    img = cv2.cvtColor(cv2.imread(str(img_paths[idx])), cv2.COLOR_BGR2RGB)
    costmap  = img_costmaps[idx]
    traj = waypoints[idx]   # (len_traj_pred, 4)
    plot_query_img_costmap_waypoints(
        query_img=img,
        costmap=costmap,
        waypoints=traj,
        step_id=idx,
        # nan_threshold=99,      # mask out outlier / unreachable cells (pl >= 99)
        use_percentile=True,   # 5th–95th %ile for cleaner colour scale
        save_path=output_imgs_dir / f"img_costmap_traj_{idx:03}.jpg"
)

In [ ]:
# Generate the video for the ffmpeg to use, and then delete the compose file

ffmpeg_compose_file = output_imgs_dir / "frames.txt"
video_path = MAP_BASE_DIR / "open-loop-viz.mp4"

# If compose file exists, delete it, otherwise it'll show up in the natsorted sequence and raise an ffmpeg error
if ffmpeg_compose_file.exists():
    ffmpeg_compose_file.unlink()

FPS_RATE = 4   # number of frames per second
DURATION_PER_FRAME = 1 / FPS_RATE

traj_imgs = natsorted(output_imgs_dir.iterdir())
# FP mode will always be creation, so as to avoid getting the output text file in the compose file
with open(ffmpeg_compose_file, "x") as f:
    for idx, img in enumerate(traj_imgs):
        f.write(f"file '{img.name}'\nduration {DURATION_PER_FRAME}\n")

cmd = [
    "ffmpeg", "-y",
    "-f", "concat", "-safe", "0",
    "-i", "frames.txt",
    "-c:v", "libx264",
    "-vf", f"fps={FPS_RATE},scale=trunc(iw/2)*2:trunc(ih/2)*2",
    "-pix_fmt", "yuv420p",
    str(video_path.resolve()),
]

try:
    subprocess.run(
        cmd, cwd=output_imgs_dir, check=True,
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
    )
except subprocess.CalledProcessError as exc:
    print(f"[create_video] FFmpeg error: {exc}")
except FileNotFoundError:
    print("[create_video] FFmpeg not found — skipping video generation.")

ffmpeg_compose_file.unlink(missing_ok=True)

In [ ]:
%matplotlib inline

# Visualize goal, which for the gostanford run is img idx 50, pixel location (160, 120)
# from PIL import Image
# img = img_paths[50]
img = cv2.cvtColor(cv2.imread(str(img_paths[50])), cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (320, 240))
plt.imshow(img)
plt.plot(160, 120, marker='*', markersize=10, markerfacecolor="yellow", markeredgecolor="brown")
plt.savefig(MAP_BASE_DIR / "outputs_viz" / "goal_img.jpg")
plt.show()
# for img in img_paths:
#     print(img)